# Passo 1 — Pré-processamento e rótulos

Pipeline sobre o dataset **real** `measured.dataset`:

1. Normalizar `input_strength` (mesmo critério do notebook 4 do Barino).
2. Filtrar $\lambda_{res} \in (1515, 1585)$ nm.
3. Gerar máscara top-$k$ ($k=4$) e converter para **classe multiclasse** (janela contígua): $C_s=\{s,\ldots,s+3\}$, $s\in\{0,\ldots,9\}$ (10 classes).
4. Verificar coerência, bijeção máscara↔classe e balanceamento.
5. Salvar artefato em `results/prepared_measured_k4.npz` (campos `y_class` e `y_mask`).

**Entrada $X$:** apenas potências normalizadas (13D). Posições FBG ficam salvas à parte para uso opcional depois.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data_utils import (
    FIGURES_DIR,
    K_DEFAULT,
    N_CLASSES,
    RANDOM_STATE,
    RESULTS_DIR,
    WL_RANGE,
    class_to_mask,
    load_measured_dataset,
    load_prepared_dataset,
    make_topk_mask,
    mask_to_window_class,
    normalize_input_strength,
    prepare_measured_classification,
    save_prepared_dataset,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(RANDOM_STATE)

K = K_DEFAULT
print(f"RANDOM_STATE={RANDOM_STATE}, k={K}, n_classes={N_CLASSES}, wl_range={WL_RANGE}")


## 1. Dados brutos (antes do pipeline)

In [ ]:
raw = load_measured_dataset()
X_raw = np.asarray(raw["input_strength"], dtype=float)
wl_raw = np.asarray(raw["wl_bragg"], dtype=float)
t_raw = np.asarray(raw["target"], dtype=float).ravel()

row_sum_raw = X_raw.sum(axis=1)
row_min_raw = X_raw.min(axis=1)

print("=== Bruto ===")
print(f"n = {X_raw.shape[0]}, n_fbgs = {X_raw.shape[1]}")
print(f"λ_res: min={t_raw.min():.6f}, max={t_raw.max():.6f} nm")
print(f"Soma por linha de input_strength: min={row_sum_raw.min():.6f}, max={row_sum_raw.max():.6f}, mean={row_sum_raw.mean():.6f}")
print(f"Mínimo por linha: min={row_min_raw.min():.6e}, max={row_min_raw.max():.6e}")
print(f"NaNs: X={np.isnan(X_raw).sum()}, wl={np.isnan(wl_raw).sum()}, target={np.isnan(t_raw).sum()}")

## 2. Pipeline: normalizar → filtrar → máscara

In [ ]:
data = prepare_measured_classification(k=K, wl_range=WL_RANGE)
X = data["X"]
y_mask = data["y_mask"]
wl = data["wl_bragg"]
target = data["target"]
keep = data["keep"]

print("=== Após pipeline ===")
print(f"n_raw = {int(data['n_raw'])}, n_kept = {int(data['n_kept'])}, removidas = {int(data['n_raw'] - data['n_kept'])}")
print(f"X shape = {X.shape}, y_mask shape = {y_mask.shape}")
print(f"λ_res filtrado: min={target.min():.6f}, max={target.max():.6f} nm")

## 3. Checagens de coerência

Só aceitamos o artefato se todas as checagens passarem.

In [ ]:
checks = {}

# Normalização
row_sum = X.sum(axis=1)
row_min = X.min(axis=1)
checks["soma_linhas_~=1"] = bool(np.allclose(row_sum, 1.0, atol=1e-10))
checks["min_linhas_~=0"] = bool(np.allclose(row_min, 0.0, atol=1e-12))
checks["X_sem_nan"] = bool(not np.isnan(X).any())
checks["X_sem_negativos"] = bool((X >= -1e-15).all())

# Filtro de faixa (aberto nas pontas, como no Barino: > e <)
checks["target_dentro_faixa"] = bool(((target > WL_RANGE[0]) & (target < WL_RANGE[1])).all())
checks["keep_conta_bate"] = bool(keep.sum() == len(target))

# Máscara: exatamente k uns por linha
ones_per_row = y_mask.sum(axis=1)
checks["exatamente_k_uns"] = bool(((ones_per_row == K).all()))

# Máscara: os k selecionados são de fato os k menores |λ_FBG - λ_res|
err = np.abs(wl - target.reshape(-1, 1))
topk_ref = np.argpartition(err, kth=K - 1, axis=1)[:, :K]
mask_ref = np.zeros_like(y_mask)
mask_ref[np.arange(len(target)).reshape(-1, 1), topk_ref] = 1
# Comparar conjuntos por linha (ordem do argpartition pode variar)
same_sets = True
for i in range(len(target)):
    if set(np.flatnonzero(y_mask[i])) != set(np.flatnonzero(mask_ref[i])):
        same_sets = False
        break
checks["mascara_coincide_topk"] = bool(same_sets)

# Empates no k-ésimo erro (pode tornar a escolha ambígua)
kth_err = np.partition(err, K - 1, axis=1)[:, K - 1]
n_at_kth = (err <= kth_err.reshape(-1, 1) + 1e-15).sum(axis=1)
n_ties = int((n_at_kth > K).sum())
checks["amostras_com_empate_no_limiar_k"] = n_ties  # informativo, não bool de falha

print("Checagens:")
for name, ok in checks.items():
    if isinstance(ok, bool):
        print(f"  [{'OK' if ok else 'FALHOU'}] {name}")
    else:
        print(f"  [info] {name} = {ok}")

bool_checks = {k: v for k, v in checks.items() if isinstance(v, bool)}
assert all(bool_checks.values()), "Alguma checagem de coerência falhou — não salvar artefato."

## 4. Balanceamento da máscara (por FBG)

In [ ]:
freq = y_mask.mean(axis=0)
count = y_mask.sum(axis=0)
bal = pd.DataFrame(
    {
        "fbg_index": np.arange(13),
        "n_positivo": count.astype(int),
        "fracao_positivo": np.round(freq, 4),
        "wl_bragg_media_nm": np.round(wl.mean(axis=0), 3),
    }
)
print(f"Esperado se uniforme: k/13 = {K/13:.4f}")
print(f"Desvio-padrão das frações: {freq.std(ddof=0):.4f}")
bal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

axes[0].bar(np.arange(13), freq, color="#4c72b0", edgecolor="black", linewidth=0.4)
axes[0].axhline(K / 13, color="C3", ls="--", label=f"uniforme k/13={K/13:.3f}")
axes[0].set_xlabel("Índice do FBG")
axes[0].set_ylabel("Fração com rótulo 1")
axes[0].set_title(f"Frequência da máscara top-{K}")
axes[0].legend(fontsize=8)

axes[1].hist(target, bins=40, color="#55a868", edgecolor="white")
axes[1].set_xlabel(r"$\lambda_{res}$ (nm)")
axes[1].set_ylabel("Contagem")
axes[1].set_title("Distribuição de $\lambda_{res}$ (filtrado)")

fig.tight_layout()
fig_path = FIGURES_DIR / "passo1_balance_mask.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Salvo: {fig_path}")

## 5. Relação máscara × $\lambda_{res}$

Para cada FBG, média de $\lambda_{res}$ nas amostras em que ele é positivo — deve acompanhar a posição média desse FBG.

In [ ]:
mean_res_when_pos = []
for j in range(13):
    sel = y_mask[:, j] == 1
    if sel.any():
        mean_res_when_pos.append(target[sel].mean())
    else:
        mean_res_when_pos.append(np.nan)
mean_res_when_pos = np.array(mean_res_when_pos)
wl_mean = wl.mean(axis=0)

rel = pd.DataFrame(
    {
        "fbg_index": np.arange(13),
        "wl_bragg_media": np.round(wl_mean, 3),
        "lambda_res_media_quando_positivo": np.round(mean_res_when_pos, 3),
        "diferenca": np.round(mean_res_when_pos - wl_mean, 3),
    }
)
rel

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(wl_mean, mean_res_when_pos, "o-", color="#4c72b0")
lims = [
    min(wl_mean.min(), np.nanmin(mean_res_when_pos)) - 2,
    max(wl_mean.max(), np.nanmax(mean_res_when_pos)) + 2,
]
ax.plot(lims, lims, "k--", lw=1, label="y = x")
for j in range(13):
    ax.text(wl_mean[j], mean_res_when_pos[j], str(j), fontsize=8, ha="left", va="bottom")
ax.set_xlabel("Posição média do FBG (nm)")
ax.set_ylabel(r"Média de $\lambda_{res}$ quando FBG=1")
ax.set_title("Coerência espacial da máscara")
ax.legend()
fig.tight_layout()
fig_path = FIGURES_DIR / "passo1_mask_vs_lambda.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Salvo: {fig_path}")

## 6. Salvar artefato

In [ ]:
out_npz = save_prepared_dataset(data)
print(f"Salvo: {out_npz} ({out_npz.stat().st_size / 1024:.1f} KB)")

# Metadados legíveis
meta = {
    "source": "paper/fbg-demodulated-lpfg/data/measured.dataset",
    "n_raw": int(data["n_raw"]),
    "n_kept": int(data["n_kept"]),
    "n_removed": int(data["n_raw"] - data["n_kept"]),
    "n_fbgs": 13,
    "k": int(K),
    "wl_range_nm": list(WL_RANGE),
    "random_state": int(RANDOM_STATE),
    "X": "input_strength normalizado (min-subtract + soma=1)",
    "y_mask": f"multi-rótulo top-{K} por |wl_bragg - target|",
    "fields_in_npz": [
        "X", "y_mask", "wl_bragg", "target", "keep",
        "X_raw_shape", "n_raw", "n_kept", "k", "wl_range", "random_state",
    ],
    "mask_positive_fraction_per_fbg": [float(f) for f in freq],
    "n_samples_with_tie_at_kth": n_ties,
    "coherence_checks_passed": True,
}
meta_path = RESULTS_DIR / "prepared_measured_k4_meta.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"Salvo: {meta_path}")

bal_path = RESULTS_DIR / "passo1_mask_balance.csv"
bal.to_csv(bal_path, index=False)
print(f"Salvo: {bal_path}")

In [ ]:
# Reload rápido para confirmar I/O
reloaded = load_prepared_dataset(out_npz)
assert reloaded["X"].shape == X.shape
assert np.array_equal(reloaded["y_mask"], y_mask)
print("Reload OK:", {k: getattr(v, "shape", v) for k, v in reloaded.items()})

## Resumo

- Artefato principal: `results/prepared_measured_k4.npz`
- Próximo passo: metodologia de avaliação (CV) sobre este artefato.
- Observação: o desbalanceamento por FBG é **esperado** se $\lambda_{res}$ se concentra no meio da faixa — não é bug da máscara.